In [ ]:
from collections import Counter
import time
import cv2
from ultralytics import YOLO


def run_yolo(source=0, model="yolo26n.pt", conf=0.45, save_to="output.mp4"):
    yolo = YOLO(model)

    # Open video source (Camera or File)
    cap = cv2.VideoCapture(
        source, cv2.CAP_DSHOW if isinstance(source, int) else cv2.CAP_ANY
    )
    if not cap.isOpened():
        return print(f"[ERROR] Cannot open source: {source}")

    w, h = int(cap.get(3)), int(cap.get(4))
    fps_in = cap.get(5) or 30.0

    # Video writer setup
    out = (
        cv2.VideoWriter(save_to, cv2.VideoWriter_fourcc(*"mp4v"), fps_in, (w, h))
        if save_to
        else None
    )

    max_counts, prev_time = {}, time.time()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Inference & annotation
        res = yolo(frame, conf=conf, verbose=False)[0]
        annotated = res.plot()

        # Count objects in current frame using Counter
        counts = Counter(res.names[int(c)] for c in res.boxes.cls)
        for name, cnt in counts.items():
            max_counts[name] = max(max_counts.get(name, 0), cnt)

        # FPS calculation
        fps = 1.0 / (time.time() - prev_time + 1e-6)
        prev_time = time.time()

        # Render UI (FPS + Object Counts)
        ui_lines = [f"FPS: {fps:.1f}"] + [f"{k}: {v}" for k, v in counts.items()]
        for i, text in enumerate(ui_lines):
            color = (0, 255, 0) if i == 0 else (255, 255, 0)
            cv2.putText(
                annotated,
                text,
                (20, 35 + i * 28),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2,
            )

        # Save and display frame
        if out:
            out.write(annotated)
        cv2.imshow("YOLO Detection", annotated)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    # Cleanup
    cap.release()
    if out:
        out.release()
    cv2.destroyAllWindows()

    # Print summary report
    print("\n" + "=" * 30 + "\n   DETECTION SUMMARY\n" + "=" * 30)
    for name, cnt in sorted(
        max_counts.items(), key=lambda x: x[1], reverse=True
    ):
        print(f"{name:<15}: {cnt}")


if __name__ == "__main__":
    run_yolo(source=0, model="yolo26n.pt", conf=0.45, save_to="output.mp4")


   DETECTION SUMMARY
person         : 2
cell phone     : 1
bottle         : 1
dining table   : 1
cup            : 1
book           : 1
